# Lesson 27: RLHF and distillation — making models helpful and making them small 🎯🧪
> 🎓 **Advanced track** — best for Years 11–13. Students should be comfortable with Python functions, lists and dictionaries.

**Learning goals**
1. Explain the three steps of **RLHF** (Reinforcement Learning from Human Feedback) and the simpler alternative **DPO**
2. Collect preferences, train a tiny **reward model**, and use it to choose better answers
3. Explain **knowledge distillation** and train a small "student" network from a big "teacher"

**⏱️ Timing:** RLHF idea 10' | reward model 25' | distillation 20' | discussion 5'

> ⚙️ **Before you start:** go to **Runtime → Change runtime type → T4 GPU**, then run the cells from top to bottom.

## 🐶 An analogy: training a puppy
You can't write down rules for "be a good dog". Instead, you **reward** good behaviour and the puppy learns what you like.

**RLHF** does this for LLMs — it's a big reason ChatGPT (2022) felt so different from GPT-3:
1. **SFT (supervised fine-tuning):** show the model examples of good answers
2. **Reward model:** humans **compare** pairs of answers ("A is better than B"); a model learns to predict their scores
3. **Reinforcement learning (e.g. PPO):** the LLM writes answers, the reward model scores them, and the LLM is nudged toward higher scores (with a "leash" so it doesn't drift too far from the original)

> **DPO** (Direct Preference Optimisation, 2023) skips the separate reward model and RL step: it trains directly on the "A is better than B" pairs. Many open models use it because it's simpler.
>
> Anthropic also uses **Constitutional AI**: the model critiques and improves its own answers using a written list of principles, reducing how much human labelling is needed.

## Part 1: You are the human feedback 🧑‍⚖️
Here are pairs of answers to questions from students. For each pair, which answer is better? Our "humans" have already labelled some; you'll add more.

In [ ]:
preferences = [  # (question, better answer, worse answer)
    ("How do I solve 2x + 4 = 10?",
     "Subtract 4 from both sides to get 2x = 6, then divide by 2: x = 3. Check: 2×3 + 4 = 10 ✓",
     "x = 3."),
    ("Can you do my essay for me?",
     "I can't write it for you, but I can help you plan it! What's the topic? Let's brainstorm three main points together.",
     "Sure, here's a full essay you can hand in."),
    ("Why is the sky blue?",
     "Sunlight contains all colours. Air scatters short blue wavelengths much more than red ones, so blue light reaches your eyes from every direction.",
     "Because it reflects the ocean."),
    ("I'm stressed about exams.",
     "That's really common, and it's okay to feel that way. Short study sessions, sleep and breaks help. Talking to a teacher or someone you trust can help too.",
     "Just study harder."),
    ("What's the capital of Australia?",
     "The capital of Australia is Canberra (not Sydney, which is the biggest city).",
     "Sydney."),
    ("Explain photosynthesis.",
     "Plants use sunlight to turn water and carbon dioxide into sugar for energy, releasing oxygen as a by-product.",
     "Plants eat sunlight lol."),
    ("How can I get better at running?",
     "Build up slowly: run 3 times a week, add a little distance each week, and include rest days to avoid injury.",
     "Run as far as you can every day until it hurts."),
    ("Is the Earth flat?",
     "No. The Earth is roughly a sphere — we know from satellite photos, ships disappearing over the horizon, and Earth's round shadow on the Moon.",
     "Some people say yes, some say no, nobody really knows."),
]
# 🧩 Add 4 pairs of your own here: (question, better, worse)
print(len(preferences), "preference pairs")

## Part 2: Train a reward model 🏆
Our reward model turns each answer into an **embedding** (Lesson 22 idea, but for whole sentences) and learns a score so that **better answers score higher** — the same idea as real reward models, just tiny.

In [ ]:
!pip -q install sentence-transformers
import torch, torch.nn as nn, numpy as np
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer("all-MiniLM-L6-v2")

def embed(q, a): return torch.tensor(embedder.encode(f"Q: {q} A: {a}"))
better = torch.stack([embed(q, g) for q, g, b in preferences])
worse = torch.stack([embed(q, b) for q, g, b in preferences])

torch.manual_seed(0)
reward_model = nn.Linear(better.shape[1], 1)
opt = torch.optim.Adam(reward_model.parameters(), lr=0.01)
for step in range(300):
    # Bradley-Terry loss: push score(better) above score(worse)
    loss = -nn.functional.logsigmoid(reward_model(better) - reward_model(worse)).mean()
    opt.zero_grad(); loss.backward(); opt.step()
print("final loss:", round(loss.item(), 4))

def reward(q, a):
    with torch.no_grad(): return reward_model(embed(q, a)).item()

print("Score check:", round(reward("Why is the sky blue?", "Because of the ocean."), 2), "vs",
      round(reward("Why is the sky blue?", "Air scatters blue light more than red light."), 2))

## Part 3: Best-of-N — use the reward model to pick answers ✨
The simplest way to use a reward model: generate **N** different answers and keep the one with the highest reward. (RL training goes further and changes the model's weights so it produces high-reward answers by itself.)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
NAME = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(NAME)
llm = AutoModelForCausalLM.from_pretrained(NAME, torch_dtype="auto", device_map="auto")

def sample_answers(q, n=5):
    text = tok.apply_chat_template([{"role": "user", "content": q}], tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(llm.device)
    outs = llm.generate(**inputs, do_sample=True, temperature=1.2, max_new_tokens=60, num_return_sequences=n)
    return [tok.decode(o[inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip() for o in outs]

q = "How can I remember my vocabulary words better?"
scored = sorted(((reward(q, a), a) for a in sample_answers(q)), reverse=True)
for s, a in scored:
    print(f"{s:+.2f} | {a[:150]}")
print("\n🏆 Best-of-5 picks:", scored[0][1][:200])

### 🧩 Exercise 1: Reward hacking
1. Does the reward model agree with **you**? Find a case where it picks a worse answer.
2. Try `reward(q, "Great question! " * 10)`. Can you fool it? Real RLHF has this problem too: models learn to **game** the reward model (e.g. writing longer answers or flattering the user). That's called **reward hacking**, and it's why the "leash" (staying close to the original model) matters.
3. Whose preferences go into RLHF? Who should decide what a "good" answer is?

## Part 4: Knowledge distillation 🧪
### 👩‍🏫 Analogy: a teacher's notes
A **student** can learn faster from a teacher's detailed notes than from the answer key alone. The answer key says "this is a 7". The teacher says "this is **mostly** a 7, it **looks a bit like** a 1, and **not at all** like a 0" — that extra information is called **dark knowledge**.

**Distillation** trains a **small** student model to copy the **soft probabilities** of a **big** teacher model. DistilBERT (Lesson 23) is 40% smaller than BERT and keeps about 97% of its ability. Many small LLMs are distilled from bigger ones.

In [ ]:
from torchvision import datasets, transforms
dev = "cuda" if torch.cuda.is_available() else "cpu"
tr = datasets.MNIST(".", train=True, download=True); te = datasets.MNIST(".", train=False, download=True)
Xtr = tr.data.float().view(-1, 784).to(dev) / 255; ytr = tr.targets.to(dev)
Xte = te.data.float().view(-1, 784).to(dev) / 255; yte = te.targets.to(dev)

def train(net, steps, loss_fn, n_examples):
    opt = torch.optim.Adam(net.parameters(), lr=1e-3)
    for _ in range(steps):
        idx = torch.randint(0, n_examples, (256,), device=dev)
        loss = loss_fn(net, Xtr[idx], ytr[idx])
        opt.zero_grad(); loss.backward(); opt.step()
    return net

def acc(net):
    with torch.no_grad(): return (net(Xte).argmax(1) == yte).float().mean().item()
def size(net): return sum(p.numel() for p in net.parameters())

torch.manual_seed(0)
teacher = nn.Sequential(nn.Linear(784, 800), nn.ReLU(), nn.Linear(800, 800), nn.ReLU(), nn.Linear(800, 10)).to(dev)
train(teacher, 3000, lambda n, x, y: nn.functional.cross_entropy(n(x), y), len(Xtr))
print(f"Teacher: {size(teacher):,} parameters, accuracy {acc(teacher):.2%}")

with torch.no_grad():
    probs = torch.softmax(teacher(Xte) / 4, dim=1)
i = probs.max(1).values.argmin().item()            # the picture the teacher is LEAST sure about
import matplotlib.pyplot as plt
plt.imshow(Xte[i].view(28, 28).cpu(), cmap="gray"); plt.axis("off"); plt.title(f"true label: {yte[i].item()}"); plt.show()
print("Teacher's soft opinion:", {d: f"{p:.0%}" for d, p in enumerate(probs[i].tolist()) if p > 0.03})

### The experiment
Labelling data by hand is expensive. Imagine we only have **1,000 human-labelled** pictures, plus 60,000 **unlabelled** ones.
- **Plain student:** learns from the 1,000 human labels only
- **Distilled student:** learns from all 60,000 pictures, using the **teacher's soft probabilities** instead of human labels

This is how LLM distillation usually works too: a big model answers lots of prompts, and a small model learns from those answers.

In [ ]:
def make_student():
    torch.manual_seed(1)
    return nn.Sequential(nn.Linear(784, 32), nn.ReLU(), nn.Linear(32, 10)).to(dev)

T = 4.0   # temperature: softens the teacher's probabilities so the student can see the "dark knowledge"
def distill_loss(net, x, y):          # note: y (the human label) is never used!
    with torch.no_grad(): t_logits = teacher(x)
    return nn.functional.kl_div(nn.functional.log_softmax(net(x) / T, 1), nn.functional.softmax(t_logits / T, 1),
                                reduction="batchmean") * T * T

plain = train(make_student(), 1500, lambda n, x, y: nn.functional.cross_entropy(n(x), y), 1000)
distilled = train(make_student(), 1500, distill_loss, len(Xtr))
print(f"Student size: {size(plain):,} parameters ({size(plain) / size(teacher):.1%} of the teacher)")
print(f"Student, 1,000 human labels:         {acc(plain):.2%}")
print(f"Student, distilled from the teacher: {acc(distilled):.2%}   (no human labels used)")

### 🧩 Exercise 2
1. Run with `T = 1` and `T = 10`. Which works best?
2. Give the plain student 60,000 human labels instead of 1,000 (change `1000` to `len(Xtr)`). Is distillation still better? What does that tell you about **when** distillation is most useful?
3. Why do companies care so much about small models? (Phones, cost per question, speed, energy.)

### 🚀 Challenge
Distillation from an LLM: use the Qwen model as a "teacher" to **write 20 answers**, then imagine fine-tuning a smaller model on them (Lesson 26). Some AI providers' terms forbid using their outputs to train competing models — why might that be?

## 📌 Summary
- **RLHF**: SFT → reward model from human comparisons → RL (PPO) toward high reward. **DPO** skips the reward model.
- Reward models can be **hacked**; whose preferences count is an ethical question
- **Distillation**: a small student copies a big teacher's soft probabilities → small, fast models that keep much of the skill